In [9]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.layers import Dense,GlobalAveragePooling2D
from tensorflow.keras.applications import MobileNet
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.mobilenet import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

In [10]:
base_model=MobileNet(weights='imagenet',include_top=False) #사전 훈련된 모델을 생성

/tmp/ipykernel_734/3510961889.py:1: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model=MobileNet(weights='imagenet',include_top=False) #사전 훈련된 모델을 생성


In [11]:
# 사전 훈련된 모델에 분류기 레이어를 추가한다
x=base_model.output
x=GlobalAveragePooling2D()(x)
x=Dense(1024,activation='relu')(x)
x=Dense(1024,activation='relu')(x)
x=Dense(512,activation='relu')(x)
preds=Dense(2,activation='softmax')(x)

model=Model(inputs=base_model.input,outputs=preds)

In [12]:
#변경되는 부분과 변경되지 않는 부분을 지정
for layer in model.layers[:20]:
  layer.trainable=False
for layer in model.layers[20:]:
  layer.trainable=True

In [13]:
#강아지와 고양이 영상을 가지고 훈련시킨다.
train_datagen=ImageDataGenerator(preprocessing_function=preprocess_input)
train_generator=train_datagen.flow_from_directory('/content/drive/MyDrive/Colab Notebooks/PetImages/',
                                                  target_size=(128,128),
                                                  color_mode='rgb',
                                                  batch_size=32,
                                                  class_mode='categorical',
                                                  shuffle=True)

Found 2200 images belonging to 2 classes.


In [14]:
model.compile(optimizer='Adam',loss='categorical_crossentropy',metrics=['accuracy'])
step_size_train=train_generator.n//train_generator.batch_size
model.fit(x=train_generator,
                    steps_per_epoch=step_size_train,
                    epochs=5)

Epoch 1/5
68/68 ━━━━━━━━━━━━━━━━━━━━ 237s 3s/step - accuracy: 0.8911 - loss: 0.4349
Epoch 2/5
68/68 ━━━━━━━━━━━━━━━━━━━━ 4s 180us/step - accuracy: 0.9375 - loss: 0.2786
Epoch 3/5


/usr/local/lib/python3.13/dist-packages/keras/src/trainers/epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


68/68 ━━━━━━━━━━━━━━━━━━━━ 103s 2s/step - accuracy: 0.9082 - loss: 0.2764
Epoch 4/5
68/68 ━━━━━━━━━━━━━━━━━━━━ 1s 162us/step - accuracy: 0.9688 - loss: 0.1025
Epoch 5/5
68/68 ━━━━━━━━━━━━━━━━━━━━ 108s 2s/step - accuracy: 0.9114 - loss: 0.2046
